In [ ]:
import math
import matplotlib.pyplot as plt

In [ ]:
class HyperLogLog:
    def __init__(self, b):
        self.b = b
        self.m = 1 << b
        self.registers = [0] * self.m
        self.alpha = 1 / (2 * math.log(2))
 
    def _rho(self, w):
        if w == 0:
            return 32 + 1
        return 32 - w.bit_length() + 1

    def add(self, item):
        h = hash(item) & 0xFFFFFFFF
        j = h >> (32 - self.b)
        w = h & ((1 << (32 - self.b)) - 1)
        self.registers[j] = max(self.registers[j], self._rho(w))

    def estimate(self):
        E = self.alpha * self.m ** 2 / sum([2 ** -r for r in self.registers])
        V = self.registers.count(0)
        if E <= 5/2 * self.m and V > 0:
            return -self.m * math.log(V / self.m)
        if E >= (1 << (32 - 5)):
            M = 1 << 32
            return -M * math.log((M - E) / M)
        return E

In [14]:
b_values = [4, 5, 6, 7, 8, 9]
n_values = list(range(1, 10001))
estimates = {b: [] for b in b_values}

for b in b_values:
    for n in n_values:
        hll = HyperLogLog(b)
        for i in range(n):
            hll.add(i)
        estimates[b].append(hll.estimate())

KeyboardInterrupt: 